In [0]:
%run ../Notebooks/00_Configuration

Configuration Loaded Successfully


In [0]:
%run ../framework/01_Utility_Functions

Configuration Loaded Successfully


In [0]:
dbutils.widgets.text("table_name", "Customer")
dbutils.widgets.text("pipeline_run_id", "")

table_name = dbutils.widgets.get("table_name")
pipeline_run_id = dbutils.widgets.get("pipeline_run_id")

print("=" * 60)
print("Generic Bronze Loader")
print("=" * 60)
print("Table Name      :", table_name)
print("Pipeline Run ID :", pipeline_run_id)

Generic Bronze Loader
Table Name      : Customer
Pipeline Run ID : 


In [0]:
from pyspark.sql.functions import col

metadata_df = (
    spark.table(f"{catalog_name}.{metadata_schema}.bronze_config")
    .filter(col("table_name") == table_name)
)

if metadata_df.count() == 0:
    raise Exception(f"No metadata found for table : {table_name}")

config = metadata_df.first()

load_strategy = config["load_strategy"]
watermark_column = config["watermark_column"]

source_folder = config["source_folder"]
target_table = config["target_table"]
file_format = config["file_format"]
primary_key = config["primary_key"]
source_system = config["source_system"]

print("=" * 60)
print("Metadata Loaded")
print("=" * 60)
print("Source Folder     :", source_folder)
print("Target Table      :", target_table)
print("File Format       :", file_format)
print("Primary Key       :", primary_key)
print("Source System     :", source_system)
print("Load Strategy     :", load_strategy)
print("Watermark Column  :", watermark_column)

Metadata Loaded
Source Folder     : customer
Target Table      : customer
File Format       : csv
Primary Key       : customer_id
Source System     : Customer_CSV
Load Strategy     : INCREMENTAL
Watermark Column  : registration_date


In [0]:
from pyspark.sql.functions import lit
from datetime import datetime

watermark_table = f"{catalog_name}.{metadata_schema}.pipeline_watermark"

watermark_df = (
    spark.table(watermark_table)
    .filter(col("table_name") == table_name)
)

if watermark_df.count() == 0:

    spark.createDataFrame(
        [(table_name, datetime(1900,1,1))],
        ["table_name","last_watermark"]
    ).write.mode("append").saveAsTable(watermark_table)

    last_watermark = datetime(1900,1,1)

else:

    last_watermark = watermark_df.first()["last_watermark"]

print("=" * 60)
print("Last Watermark")
print("=" * 60)
print(last_watermark)

Last Watermark
2023-04-16 18:55:40


In [0]:
source_path = landing_path + source_folder

print("=" * 60)
print("Landing Path")
print("=" * 60)
print(source_path)

Landing Path
abfss://landing@adlsinsurancedevstorage.dfs.core.windows.net/customer


In [0]:
df = (
    spark.read
        .format(file_format)
        .option("header", True)
        .load(source_path)
)

rows_before = df.count()

print("=" * 60)
print("Rows Read From Landing :", rows_before)

Rows Read From Landing : 1050


In [0]:
from pyspark.sql.functions import col, to_timestamp

# COMMAND ----------
# CELL 8 - Apply Load Strategy

from pyspark.sql.functions import col, to_timestamp, lit

print("=" * 60)
print("Applying Load Strategy")
print("=" * 60)

if (
    load_strategy is not None
    and load_strategy.upper() == "INCREMENTAL"
    and watermark_column is not None
    and watermark_column != ""
):

    print("Load Type : INCREMENTAL")
    print("Watermark :", last_watermark)

    # Create a temporary timestamp column only for filtering
    df = (
        df.withColumn(
            "_watermark_ts",
            to_timestamp(col(watermark_column))
        )
    )
     
    # Keep only new records
    df = (
        df.filter(
            col("_watermark_ts") > lit(last_watermark)
        )
    )

    # Remove temporary column before writing to Bronze
    df = df.drop("_watermark_ts")

else:

    print("Load Type : FULL")

rows_read = df.count()

print("Rows After Filter :", rows_read)

display(df.limit(10)) 

Applying Load Strategy
Load Type : INCREMENTAL
Watermark : 2023-04-16 18:55:40
Rows After Filter : 0


customer_id,first_name,last_name,email,phone,country,city,registration_date,date_of_birth,gender


In [0]:
duplicate_count = (
    df.groupBy(primary_key)
      .count()
      .filter("count > 1")
      .count()
)

null_key_count = (
    df.filter(col(primary_key).isNull())
      .count()
)

print("=" * 60)
print("Data Quality")
print("=" * 60)
print("Rows Read       :", rows_read)
print("Duplicate Keys  :", duplicate_count)
print("Null Keys       :", null_key_count)

Data Quality
Rows Read       : 0
Duplicate Keys  : 0
Null Keys       : 0


In [0]:
df = add_metadata(
    df,
    source_system
)

display(df)

customer_id,first_name,last_name,email,phone,country,city,registration_date,date_of_birth,gender,ingestion_timestamp,source_file_name,load_id,pipeline_run_id,source_system


In [0]:
bronze_table = (
    f"{catalog_name}."
    f"{bronze_schema}."
    f"{target_table}"
)

print("=" * 60)
print("Bronze Table")
print("=" * 60)
print(bronze_table)

Bronze Table
dbw_insurance.insurance_bronze.customer


In [0]:
load_type = load_strategy.strip().upper()

write_mode = (
    "overwrite"
    if load_type in ["FULL", "FULL_LOAD"]
    else "append"
)

(
    df.write
      .format("delta")
      .mode(write_mode)
      .saveAsTable(bronze_table)
)

# Number of rows written in THIS run
rows_written = rows_read

# Current total rows in Bronze
bronze_total_rows = spark.table(bronze_table).count()

print("=" * 60)
print("Bronze Load Completed")
print("=" * 60)

print("Write Mode          :", write_mode)
print("Rows Written        :", rows_written)
print("Bronze Total Rows   :", bronze_total_rows)

Bronze Load Completed
Write Mode          : append
Rows Written        : 0
Bronze Total Rows   : 2796


In [0]:
# 
if (
    load_type == "INCREMENTAL"
    and rows_written > 0
):

    new_watermark = (
        df
        .withColumn(
            "_watermark_ts",
            to_timestamp(col(watermark_column))
        )
        .agg({"_watermark_ts": "max"})
        .collect()[0][0]
    )

    spark.sql(f"""
        UPDATE {catalog_name}.{metadata_schema}.pipeline_watermark
        SET last_watermark = TIMESTAMP('{new_watermark}')
        WHERE table_name = '{table_name}'
    """)

    print("=" * 60)
    print("Watermark Updated")
    print("=" * 60)
    print(new_watermark)

else:

    print("=" * 60)
    print("Watermark Not Updated")

Watermark Not Updated


In [0]:
# pipeline_name = "Generic_Bronze_Loader"

# write_audit(
#     pipeline_name=pipeline_name,
#     table_name=target_table,
#     load_type=load_strategy,
#     rows_read=rows_read,
#     rows_written=rows_written,
#     status="SUCCESS",
#     error_message="",
#     pipeline_run_id=pipeline_run_id
# )

write_audit(
    pipeline_name="Generic_Bronze_Loader",
    table_name=target_table,
    load_type=load_strategy,
    rows_read=int(rows_read),
    rows_written=int(rows_written),
    status="SUCCESS",
    error_message="",
    pipeline_run_id=pipeline_run_id
)

In [0]:
%sql
dESCRIBE TABLE dbw_insurance.insurance_audit.pipeline_audit;

col_name,data_type,comment
pipeline_name,string,null
table_name,string,null
load_type,string,null
start_time,timestamp,null
end_time,timestamp,null
rows_read,int,null
rows_written,int,null
status,string,null
error_message,string,null
pipeline_run_id,string,null


In [0]:
# print("=" * 60)
# print("GENERIC BRONZE LOAD COMPLETED")
# print("=" * 60)

# print("Table           :", table_name)
# print("Load Strategy   :", load_strategy)
# print("Rows Read       :", rows_read)
# print("Rows Written    :", rows_written)
# print("Pipeline Run ID :", pipeline_run_id)

print("=" * 70)
print("GENERIC BRONZE LOADER COMPLETED")
print("=" * 70)

print(f"Table Name         : {table_name}")
print(f"Load Strategy      : {load_strategy}")
print(f"Watermark Column   : {watermark_column}")
print(f"Rows Read          : {rows_read}")
print(f"Rows Written       : {rows_written}")
print(f"Bronze Total Rows  : {bronze_total_rows}")
print(f"Pipeline Run ID    : {pipeline_run_id}")

print("=" * 70)

GENERIC BRONZE LOADER COMPLETED
Table Name         : Customer
Load Strategy      : INCREMENTAL
Watermark Column   : registration_date
Rows Read          : 0
Rows Written       : 0
Bronze Total Rows  : 2796
Pipeline Run ID    : 


In [0]:
print("=" * 70)
print("GENERIC BRONZE LOADER COMPLETED")
print("=" * 70)

print(f"Table Name         : {table_name}")
print(f"Load Strategy      : {load_strategy}")
print(f"Watermark Column   : {watermark_column}")
print(f"Rows Read          : {rows_read}")
print(f"Rows Written       : {rows_written}")
print(f"Bronze Total Rows  : {bronze_total_rows}")
print(f"Pipeline Run ID    : {pipeline_run_id}")

print("=" * 70)

GENERIC BRONZE LOADER COMPLETED
Table Name         : Customer
Load Strategy      : INCREMENTAL
Watermark Column   : registration_date
Rows Read          : 0
Rows Written       : 0
Bronze Total Rows  : 2796
Pipeline Run ID    : 


In [0]:
%sql
SELECT
    load_strategy,
    watermark_column
FROM insurance_metadata.bronze_config
WHERE table_name = 'Customer';

load_strategy,watermark_column
INCREMENTAL,registration_date
